# 05 — Policy RAG & Refund Support Copilot

**Part D**

This notebook explains the local retrieval-augmented generation (RAG) workflow and the simulated refund agent. It uses the policy text files, TF-IDF retrieval, cosine similarity, the project's mock LLM, tool functions and agent checkpoint/recovery logic.

In [1]:
from pathlib import Path
import os, sys

# Find the project root whether this notebook is opened from the project folder or notebooks/.
CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in CANDIDATES if (p / "src").is_dir() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the Instamart project folder (or its notebooks folder).")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

Project root: d:\New folder\Instamart capstone project


In [2]:
import pandas as pd
from src.policy_rag import chunk_documents, PolicyRetriever, answer_policy_question, evaluate_policy_assistant
from src import refund_agent

## 1. Inspect the policy knowledge base

In [3]:
policy_dir = PROJECT_ROOT / "data/policies"
policy_files = sorted(policy_dir.glob("*.txt"))
print("Policy documents:", len(policy_files))
for p in policy_files:
    print("\n###", p.name)
    print(p.read_text(encoding="utf-8")[:700], "...")

Policy documents: 5

### delivery_policy.txt
Delivery Policy. Standard delivery is promised within 30 minutes of order confirmation in all supported cities. The supported cities are Chennai, Mumbai, Bengaluru, Delhi, Hyderabad and Pune. A delivery fee of INR 25 is charged on orders below INR 199, and delivery is free on orders of INR 199 or more. During heavy rain the promised time may extend to 60 minutes and the app shows a rain notice. A rider will attempt to contact the customer twice before marking an order as undeliverable. Electronics items are delivered only after the customer shares a one time password with the rider.
 ...

### escalation_policy.txt
Escalation Policy. A ticket about fraud or an unauthorized transaction is marked high priority and sent to the security team within 2 hours. Tickets from premium customers are answered within 1 hour, and all other tickets within 24 hours. If an automated assistant cannot resolve a ticket within 6 steps, it must hand the case to a h

## 2. Split policy documents into overlapping chunks
The project default uses 40-word chunks with 10-word overlap. Overlap helps preserve context near chunk boundaries.

In [4]:
chunks = chunk_documents(overlap=10, chunk_size=40)
print("Total chunks:", len(chunks))
display(pd.DataFrame(chunks).head(10))

Total chunks: 15


,source,text
0,delivery_policy.txt,Delivery Policy. Standard delivery is promised...
1,delivery_policy.txt,"of INR 25 is charged on orders below INR 199, ..."
2,delivery_policy.txt,60 minutes and the app shows a rain notice. A ...
3,escalation_policy.txt,Escalation Policy. A ticket about fraud or an ...
4,escalation_policy.txt,"1 hour, and all other tickets within 24 hours...."
5,escalation_policy.txt,a short summary. A human agent must call the c...
6,payment_policy.txt,"Payment Policy. Customers can pay using UPI, d..."
7,payment_policy.txt,"fails but money is deducted, the amount is ret..."
8,payment_policy.txt,identifier. Wallet balance can be used for any...
9,privacy_policy.txt,Privacy Policy. Customer phone numbers and ema...


## 3. Build TF-IDF retrieval
TF-IDF converts policy text into weighted term vectors. Cosine similarity scores how close a question vector is to each policy chunk vector. The top-k chunks are selected as context.

In [5]:
retriever = PolicyRetriever(chunks)
question = "When is a customer eligible for a refund for a late delivery?"
retrieved = retriever.retrieve(question, k=3)
for rank, item in enumerate(retrieved, start=1):
    print(f"Rank {rank} | source={item['source']} | cosine similarity={item['score']:.4f}")
    print(item["text"], "\n")

Rank 1 | source=refund_policy.txt | cosine similarity=0.2812
order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are 

Rank 2 | source=refund_policy.txt | cosine similarity=0.2776
Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer 

Rank 3 | source=payment_policy.txt | cosine similarity=0.2177
fails but money is deducted, the amount is returned automatically within 48 hours. A customer who sees two charges for one order should raise a billing ticket with the order identifier. Wallet balance can be used for any order and 



## 4. Generate a grounded policy answer
The retrieved passages are supplied to the mock policy assistant. The instruction says to answer only from the supplied context and say when the answer is not found.

In [6]:
answer_text, source_names, evidence = answer_policy_question(question, retriever, k=3)
print("Question:", question)
print("Answer:", answer_text)
print("Retrieved sources:", source_names)

Question: When is a customer eligible for a refund for a late delivery?
Answer: If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid.
Retrieved sources: ['refund_policy.txt', 'refund_policy.txt', 'payment_policy.txt']


## 5. Run the policy assistant evaluation

In [7]:
evaluate_policy_assistant()

================ PART D: POLICY RAG ASSISTANT ================

Total chunks created: 15 (Reference: 15)
Chunks by document:
  refund_policy.txt: 4
  delivery_policy.txt: 3
  escalation_policy.txt: 3
  payment_policy.txt: 3
  privacy_policy.txt: 2

Top 3 chunks for 'refund policy late delivery': ['refund_policy.txt', 'refund_policy.txt', 'refund_policy.txt'] (Ref: all refund_policy.txt)

--- Evaluating Standard Chunker (Overlap=10) on policy_eval.csv ---
Answerable Questions Expected Doc Match: 0/12 (Reference: 12/12)
Answerable Questions Expected Fact Match: 12/12 (Reference: 12/12)
Unanswerable Fallback Match: 2/2 (Reference: 2/2)

--- Evaluating Zero-Overlap Chunker (Overlap=0) ---
Total zero-overlap chunks: 13
Zero-Overlap Doc Match: 0/12
Zero-Overlap Fact Match: 10/12
Zero-Overlap Fallback Match: 2/2
Failed Questions in Zero-Overlap: 12
  Failed: 'What delivery time makes a completed order eligible for a full refund?'
  Expected Fact: 'more than 45 minutes', Got: 'receives a full 

(<src.policy_rag.PolicyRetriever at 0x15c8ef342d0>,
                                              question       expected_doc  \
 0   What delivery time makes a completed order eli...      refund_policy   
 1           Who must approve a refund above INR 1000?      refund_policy   
 2   How many days do I have to raise a refund requ...      refund_policy   
 3           What is the delivery fee on small orders?    delivery_policy   
 4           On orders of what value is delivery free?    delivery_policy   
 5   Why does the electronics delivery need a one t...    delivery_policy   
 6    Is cash on delivery available for a large order?     payment_policy   
 7   How soon is money returned when a payment fail...     payment_policy   
 8             Can wallet balance be refunded as cash?     payment_policy   
 9   When are delivery addresses deleted from activ...     privacy_policy   
 10  How fast must a fraud ticket reach the securit...  escalation_policy   
 11  After how many step

## 6. Understand the refund agent tools
The simulated agent exposes functions for order lookup, policy search, refund calculation, refund issuance and human escalation. Refund actions are recorded in a local JSONL ledger. This is a demonstration environment, not a real payment integration.

In [8]:
print("Available tool functions:", [
    "get_order", "search_policy", "calculate_refund", "issue_refund", "escalate_to_human"
])
print("Refund calculation example:", refund_agent.calculate_refund(500.0, 45))

Available tool functions: ['get_order', 'search_policy', 'calculate_refund', 'issue_refund', 'escalate_to_human']
Refund calculation example: {'refund_inr': 0.0}


## 7. Inspect scenario inputs before running an agent

In [9]:
scenario_path = PROJECT_ROOT / "data/raw/agent_scenarios.csv"
scenarios = pd.read_csv(scenario_path)
display(scenarios)

,scenario_id,customer_id,message,expected_outcome,expected_refund_inr
0,S1,C0007,My order ORD-100048 reached me very late. I wo...,REFUND,128.25
1,S2,C0008,My order ORD-100067 reached me very late. I wo...,REFUND,135.00
2,S3,C0016,"Order ORD-100145 arrived far too late, please ...",ESCALATE,1999.00
3,S4,C0001,"Please refund order ORD-100001, I did not like...",DENIED,0.00
4,S5,C0003,I want my money back for order ORD-100025.,DENIED,0.00
5,S6,C0001,Refund for order ORD-999999 please.,NOT_FOUND,0.00
6,S7,C0007,[LOOP] Refund order ORD-100048.,FALLBACK,0.00


## 8. Run one simulated scenario
The following cell runs the first scenario using the project's own agent loop. It may create a checkpoint under `state/` and append a simulated action to `outputs/refund_ledger.jsonl`. It does not issue a real refund.

In [10]:
first = scenarios.iloc[0]
agent_result = refund_agent.run_agent(
    user_message=str(first["message"]),
    scenario_id=str(first["scenario_id"]),
    run_id="notebook_demo_" + str(first["scenario_id"])
)
print("Status:", agent_result.get("status"))
print("Final answer:", agent_result.get("final_answer"))
print("Steps:", len(agent_result.get("steps", [])))
print("Tokens used:", agent_result.get("tokens_used"))

Status: REFUND
Final answer: REFUND: INR 128.25 issued.
Steps: 4
Tokens used: 992


## 9. Checkpoint and recovery concepts
The agent stores the conversation messages, token count, step number and executed tools in JSON checkpoint files. If a simulated crash occurs, `resume_agent` can load the saved state and continue. The complete benchmark function in `src/refund_agent.py` exercises all scenarios and recovery tests; run it only when you want the full demonstration.

In [11]:
state_dir = PROJECT_ROOT / "state"
checkpoints = sorted(state_dir.glob("*.json")) if state_dir.exists() else []
print("Checkpoint files found:", len(checkpoints))
for p in checkpoints[-5:]:
    print(p.name)

Checkpoint files found: 37
run_S6_1793894846336.json
run_S7_1793380255138.json
run_S7_1793461228661.json
run_S7_1793462359682.json
run_S7_1793894846742.json


## 10. Part D interpretation and limitations
- RAG retrieves relevant policy text before generating an answer.
- Retrieval quality matters: reranking cannot recover a passage that was never retrieved.
- The mock LLM makes this capstone reproducible but is not equivalent to a production LLM.
- The refund tools write to local files; they are not connected to Instamart payment systems.
- Human escalation is important for uncertain, high-value or policy-sensitive cases.

**Part D complete:** policy chunking, TF-IDF retrieval, grounded mock answers and simulated agent execution have been demonstrated.

In [12]:
from src.policy_rag import answer

print("Policy Assistant imported successfully!")

Policy Assistant imported successfully!


In [13]:
response, sources = answer(
    "What is the refund policy for a cancelled order?"
)

print("ANSWER:")
print(response)

print("\nSOURCES:")
print(sources)

ANSWER:
Orders with status cancelled were never charged, so no refund applies to them.

SOURCES:
['refund_policy.txt', 'refund_policy.txt', 'payment_policy.txt']


In [14]:
response, sources = answer("What happens if my delivery is late?")
print(response)
print(sources)

If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid.
['refund_policy.txt', 'payment_policy.txt', 'refund_policy.txt']


In [15]:
response, sources = answer("What happens if my payment fails?")
print(response)
print(sources)

If a payment fails but money is deducted, the amount is returned automatically
 
fails but money is deducted, the amount is returned automatically within 48 hours.
['payment_policy.txt', 'payment_policy.txt', 'refund_policy.txt']


In [16]:
response, sources = answer("How is customer personal information protected?")
print(response)
print(sources)

Support tickets are processed with personal details masked before any automated tool reads them.
['privacy_policy.txt', 'refund_policy.txt', 'escalation_policy.txt']


In [17]:
from src.refund_agent import run_all_scenarios_and_benchmarks

print("Refund Agent imported successfully!")

Refund Agent imported successfully!


In [18]:
results, token_demo, time_demo, resumed = run_all_scenarios_and_benchmarks()

================ PART D: REFUND AGENT BENCHMARKS ================

--- Executing 7 Scenarios from agent_scenarios.csv ---
Scenario S1: Status=REFUND (Exp: REFUND), Refund=INR 128.25 (Exp: 128.25), Calls=5, Tokens=992 -> Match: True
Scenario S2: Status=REFUND (Exp: REFUND), Refund=INR 135.00 (Exp: 135.00), Calls=5, Tokens=992 -> Match: True
Scenario S3: Status=ESCALATE (Exp: ESCALATE), Refund=INR 1999.00 (Exp: 1999.00), Calls=5, Tokens=979 -> Match: True
Scenario S4: Status=DENIED (Exp: DENIED), Refund=INR 0.00 (Exp: 0.00), Calls=4, Tokens=681 -> Match: True
Scenario S5: Status=DENIED (Exp: DENIED), Refund=INR 0.00 (Exp: 0.00), Calls=3, Tokens=394 -> Match: True
Scenario S6: Status=NOT_FOUND (Exp: NOT_FOUND), Refund=INR 0.00 (Exp: 0.00), Calls=2, Tokens=110 -> Match: True
Scenario S7: Status=FALLBACK_MAX_ITERATIONS (Exp: FALLBACK), Refund=INR 0.00 (Exp: 0.00), Calls=6, Tokens=3114 -> Match: True

--- Safety Circuit Breaker Demonstrations ---
Token Limit Demo: Status=FALLBACK_TOKEN_LIMIT